# SWAP-Stress: inference (stages 05-07)

From a trained model and a day of SMAP L3 soil moisture to the released product.

1. The feature contract — what the model requires, what the raster stack supplies
2. SMAP L3 coverage (stage 05's dynamic input)
3. Stage 05 — daily prediction
4. Stage 06 — gap fill, before and after
5. Spatial sanity: one month of Level 2
6. Temporal sanity at three sites
7. Stage 07 — the released bands

Every path resolves from the same run configs the stages read, so this notebook
tours the run the release was actually built from. Nothing here recomputes what a
stage does: prediction, gap fill, and band derivation are library calls.

In [ ]:
from __future__ import annotations

import glob
import os
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import rasterio

from swapstress.config import load_config
from swapstress.figures import basemap

# ---------------------------------------------------------------------------
# The one thing you may need to change: where this repo is checked out. Every
# data path below comes out of the run configs, not out of this notebook.
# ---------------------------------------------------------------------------
REPO = os.path.abspath(os.environ.get("SWAPSTRESS_REPO", "."))

predict_cfg = load_config(
    os.path.join(REPO, "configs", "predict_9km_global_pruned.toml"), {}
)
gapfill_cfg = load_config(
    os.path.join(REPO, "configs", "gapfill_9km_global_pruned.toml"), {}
)

MODEL_DIR = predict_cfg["model_dir"]
FEAT_DIR = predict_cfg["static_dir"]
SMAP_DIR = predict_cfg["smap_dir"]
PREFIX = gapfill_cfg["prefix"]


def has_rasters(directory):
    return bool(glob.glob(os.path.join(directory, f"{PREFIX}_*.tif")))


def stage_dir(declared):
    """The directory a stage's config declares, or the sibling the run wrote to.

    The shipped release ran the L3 and L4 theta inputs side by side and gave each
    variant its own directory, so the nominal path in the config can be empty
    while the run that shipped sits in ``<dir>_l3``. Only a sibling that actually
    holds rasters is accepted, and the choice is printed rather than assumed.
    """
    if has_rasters(declared):
        return declared
    for suffix in ("_l3",):
        candidate = declared + suffix
        if has_rasters(candidate):
            print(f"{declared} is empty; using {candidate} instead")
            return candidate
    return declared


LEVEL1_DIR = stage_dir(predict_cfg["output_dir"])  # stage 05: direct retrievals
LEVEL2_DIR = stage_dir(gapfill_cfg["output_dir"])  # stage 06: gap-filled

HAVE_LEVEL1 = has_rasters(LEVEL1_DIR)
HAVE_LEVEL2 = has_rasters(LEVEL2_DIR)

# Prediction is expensive even for one day (it loads the RF and the whole static
# stack), so it stays off until you ask for it.
RUN_PREDICT = False

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# CONUS outlines for the maps. The product is global; the maps zoom to CONUS
# because that is where the in-situ sources are and where the eye can check it.
states = basemap.load_conus_states(crs=6933)

print("MODEL_DIR: ", MODEL_DIR)
print("FEAT_DIR:  ", FEAT_DIR)
print("SMAP_DIR:  ", SMAP_DIR)
print(
    "LEVEL1_DIR:", LEVEL1_DIR, "[has rasters]" if HAVE_LEVEL1 else "[stage 05 not run]"
)
print(
    "LEVEL2_DIR:", LEVEL2_DIR, "[has rasters]" if HAVE_LEVEL2 else "[stage 06 not run]"
)

In [ ]:
def draw_map(
    ax,
    arr,
    extent,
    cmap="viridis",
    vmin=None,
    vmax=None,
    title=None,
    label=None,
    zoom_conus=True,
):
    """Draw one EASE-Grid array with state outlines. Display plumbing only."""
    valid = arr[np.isfinite(arr)]
    if vmin is None or vmax is None:
        vmin, vmax = np.percentile(valid, [2, 98]) if valid.size else (0, 1)
    im = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)
    states.boundary.plot(ax=ax, color="0.3", linewidth=0.3)
    if zoom_conus:
        x0, y0, x1, y1 = states.total_bounds
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
    else:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
    ax.figure.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label=label)
    ax.set_title(title or "", fontsize=11)
    ax.tick_params(labelsize=7)
    return im


def read_day(path, nodata=-9999.0):
    """Read band 1 as float32 with nodata as NaN, plus the plotting extent."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        fill = src.nodata if src.nodata is not None else nodata
        if fill is not None and np.isfinite(fill):
            arr[arr == fill] = np.nan
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    return arr, extent


def pick_date(rasters, preferred):
    """The preferred date if the record has it, else the middle of the record."""
    days = sorted(rasters)
    return preferred if preferred in rasters else days[len(days) // 2]


print("helpers ready")

## 1) The feature contract

The model artifact carries its own ordered feature list, and stage 05 refuses to
run unless the raster stack supplies every static feature in it.
`validate_feature_contract` is the check: it splits the saved features into the
ones that must come from rasters and the two that are set per run (`depth_cm`,
`rosetta_level`), and it fails loudly if `theta` — the one dynamic input — is
missing.

`StaticRasterStack.load` then does the matching against the rasters. It is not
called here because it holds the whole stack in memory; the cross-check below
reads band names only.

Loading the artifact pulls the saved forest in, which is a couple of gigabytes.

In [ ]:
from swapstress.inference.predict import ModelArtifacts, validate_feature_contract

artifacts = ModelArtifacts.load(MODEL_DIR)
static_features, fixed_features = validate_feature_contract(artifacts)

print(f"model features: {len(artifacts.feature_names)}")
print(f"  from rasters: {len(static_features)}")
print(f"  set per run:  {sorted(fixed_features)}")
print("  dynamic:      theta (SMAP L3 soil moisture)")
print("\nfirst 10:", artifacts.feature_names[:10])

In [ ]:
# Which file in the static stack supplies each required feature.
supplied = {}
for fname in sorted(f for f in os.listdir(FEAT_DIR) if f.endswith("_ease2.tif")):
    with rasterio.open(os.path.join(FEAT_DIR, fname)) as src:
        for i in range(src.count):
            name = src.descriptions[i]
            if name in static_features:
                supplied.setdefault(name, []).append(f"{fname}:{i + 1}")

missing = sorted(static_features - set(supplied))
duplicated = {k: v for k, v in supplied.items() if len(v) > 1}

print(f"required static features: {len(static_features)}")
print(f"found in the stack:       {len(supplied)}")
print(f"missing:                  {missing if missing else 'none'}")
print(f"supplied more than once:  {duplicated if duplicated else 'none'}")

## 2) SMAP L3 coverage

`theta` comes from SMAP L3 (SPL3SMP_E) enhanced 9 km retrievals. One pass leaves
most of the grid unobserved on any given day, which is the whole reason stage 06
exists.

**L3, not L4.** L4 assimilates brightness temperatures into a land surface model
that already carries pedotransfer-derived hydraulic parameters. Feeding it to
this model would mean learning to invert those assumptions rather than the
observed theta-to-psi relationship, so L4 is excluded by policy — from the
features and from theta.

`swapstress.figures.fig03_coverage.compute_coverage` is the same accumulator the
descriptor's coverage figure uses. It opens one raster at a time and keeps only
the counts.

In [ ]:
from swapstress.figures.fig03_coverage import compute_coverage

smap_cov = compute_coverage(SMAP_DIR, prefix="smap_sm")

daily = np.asarray(smap_cov.daily_counts, dtype=float)
observed = daily[daily > 0]
total_px = smap_cov.valid_days.size

print(f"days in span:  {smap_cov.n_days:,}  ({smap_cov.absent_days:,} with no file)")
print(
    f"median valid px:   {np.median(observed):,.0f} / {total_px:,} "
    f"({100 * np.median(observed) / total_px:.1f}%)"
)

fig, ax = plt.subplots(figsize=(14, 4), dpi=120)
ax.plot(smap_cov.calendar, daily, lw=0.6, color="steelblue", alpha=0.85)
ax.axhline(
    np.median(observed),
    color="tomato",
    lw=1.2,
    ls="--",
    label=f"median {np.median(observed):,.0f} px",
)
ax.set_xlabel("Date")
ax.set_ylabel("Valid pixels")
ax.set_title("SMAP L3 valid retrievals per day")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
from swapstress.inference.gapfill import discover_source_rasters

smap_rasters = discover_source_rasters(SMAP_DIR, prefix="smap_sm")
smap_day = pick_date(smap_rasters, date(2024, 6, 15))
arr, extent = read_day(smap_rasters[smap_day])

fig, ax = plt.subplots(figsize=(11, 5), dpi=120)
draw_map(
    ax,
    arr,
    extent,
    cmap="viridis",
    label="VWC (m$^3$/m$^3$)",
    title=f"SMAP L3 soil moisture — {smap_day} "
    f"({int(np.isfinite(arr).sum()):,} valid px globally)",
)
fig.tight_layout()
plt.show()

## 3) Stage 05 — prediction

One raster per day of `log10_suction_cm`, written only where SMAP returned a
retrieval:

```bash
uv run swapstress-predict --config configs/predict_9km_global_pruned.toml
```

`--dry-run` reports the model directory, static stack, SMAP directory, and output
directory without loading anything. The config sets `depth_cm`, `batch_size`
(pixels per RF batch — memory against speed), and `write_linear`; any of them can
be overridden on the command line.

The cell below calls `run_prediction` directly for a single day. It is off by
default because it loads the RF and the full static stack.

In [ ]:
if RUN_PREDICT:
    from swapstress.inference.predict import run_prediction

    demo = "20240615"
    run_prediction(
        model_dir=MODEL_DIR,
        static_dir=FEAT_DIR,
        smap_dir=SMAP_DIR,
        output_dir=LEVEL1_DIR,
        start_date=demo,
        end_date=demo,
        depth_cm=predict_cfg["depth_cm"],
        rosetta_level=None,
        batch_size=predict_cfg["batch_size"],
        overwrite=False,
        write_linear=predict_cfg["write_linear"],
    )

    arr, extent = read_day(os.path.join(LEVEL1_DIR, f"suction_{demo}.tif"))
    fig, ax = plt.subplots(figsize=(11, 5), dpi=120)
    draw_map(
        ax,
        arr,
        extent,
        cmap="RdBu",
        label="log10 suction (cm)",
        title=f"Predicted suction — {demo}",
    )
    fig.tight_layout()
    plt.show()
else:
    print("RUN_PREDICT=False — skipping the stage 05 demo")

## 4) Stage 06 — gap fill

Level 1 inherits SMAP's coverage. Stage 06 interpolates each pixel's own time
series across the days it was not observed, so Level 2 is complete over the
record's span:

```bash
uv run swapstress-gapfill --config configs/gapfill_9km_global_pruned.toml
```

The fill is per pixel and temporal only — no spatial borrowing — so a filled
value never invents structure the pixel's own record does not support. Stage 07
carries a `gapfill_flag` band so a filled value is always distinguishable from a
retrieved one.

In [ ]:
level1_cov = compute_coverage(LEVEL1_DIR, prefix=PREFIX)
l1 = np.asarray(level1_cov.daily_counts, dtype=float)
print(f"Level 1 — median valid px/day: {np.median(l1[l1 > 0]):,.0f}")

fig, ax = plt.subplots(figsize=(14, 4), dpi=120)
if HAVE_LEVEL2:
    level2_cov = compute_coverage(LEVEL2_DIR, prefix=PREFIX)
    l2 = np.asarray(level2_cov.daily_counts, dtype=float)
    print(f"Level 2 — median valid px/day: {np.median(l2[l2 > 0]):,.0f}")
    ax.fill_between(
        level2_cov.calendar,
        l2,
        color="tomato",
        alpha=0.35,
        label="Level 2 (gap-filled)",
    )
else:
    print(
        f"No Level 2 rasters in {LEVEL2_DIR} — stage 06 has not been run for "
        "this release. The comparison panels below stay empty until it is."
    )

ax.fill_between(
    level1_cov.calendar, l1, color="steelblue", alpha=0.75, label="Level 1 (retrieved)"
)
ax.set_xlabel("Date")
ax.set_ylabel("Valid pixels")
ax.set_title("Valid pixels per day: Level 1 against Level 2")
ax.legend()
fig.tight_layout()
out = os.path.join(OUT_DIR, "07_gapfill_coverage.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

In [ ]:
level1 = discover_source_rasters(LEVEL1_DIR, prefix=PREFIX)
level2 = discover_source_rasters(LEVEL2_DIR, prefix=PREFIX) if HAVE_LEVEL2 else {}

show_day = pick_date(level1, date(2024, 6, 15))
l1_arr, extent = read_day(level1[show_day])

if show_day in level2:
    l2_arr, _ = read_day(level2[show_day])
    both = np.concatenate([l1_arr[np.isfinite(l1_arr)], l2_arr[np.isfinite(l2_arr)]])
    vmin, vmax = np.percentile(both, [2, 98])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), dpi=120)
    draw_map(
        ax1,
        l1_arr,
        extent,
        cmap="RdBu",
        vmin=vmin,
        vmax=vmax,
        label="log10 suction (cm)",
        title=f"Level 1 — {show_day} ({int(np.isfinite(l1_arr).sum()):,} px)",
    )
    draw_map(
        ax2,
        l2_arr,
        extent,
        cmap="RdBu",
        vmin=vmin,
        vmax=vmax,
        label="log10 suction (cm)",
        title=f"Level 2 — {show_day} ({int(np.isfinite(l2_arr).sum()):,} px)",
    )
else:
    fig, ax1 = plt.subplots(figsize=(11, 5), dpi=120)
    draw_map(
        ax1,
        l1_arr,
        extent,
        cmap="RdBu",
        label="log10 suction (cm)",
        title=f"Level 1 — {show_day} ({int(np.isfinite(l1_arr).sum()):,} px)",
    )
    print(f"No Level 2 raster for {show_day}; showing Level 1 alone.")

fig.tight_layout()
plt.show()

## 5) Spatial sanity — one month of Level 2

Pixel-wise mean and standard deviation over a single month. What to expect: high
suction through the arid West, low suction in the humid East, the Rockies
legible as terrain. The standard deviation is the more diagnostic panel — it
should be largest where storms actually cycle the profile, not uniform.

In [ ]:
# Level 2 if stage 06 has been run, otherwise the same statistic over the days a
# pixel was actually retrieved.
record = level2 if level2 else level1
label = "Level 2" if level2 else "Level 1"

month = (2024, 7)
month_days = [d for d in sorted(record) if (d.year, d.month) == month]
if not month_days:
    latest = sorted(record)[-1]
    month = (latest.year, latest.month)
    month_days = [d for d in sorted(record) if (d.year, d.month) == month]
print(f"{month[0]}-{month[1]:02d}: {len(month_days)} {label} rasters")

first, extent = read_day(record[month_days[0]])
stack = np.full((len(month_days), *first.shape), np.nan, dtype=np.float32)
stack[0] = first
for i, d in enumerate(month_days[1:], start=1):
    stack[i], _ = read_day(record[d])

mean_suction = np.nanmean(stack, axis=0)
std_suction = np.nanstd(stack, axis=0)
del stack

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), dpi=120)
draw_map(
    ax1,
    mean_suction,
    extent,
    cmap="RdBu_r",
    label="log10 suction (cm)",
    title=f"{label} mean — {month[0]}-{month[1]:02d}",
)
draw_map(
    ax2,
    std_suction,
    extent,
    cmap="RdBu_r",
    label="log10 suction (cm)",
    title=f"{label} std dev — {month[0]}-{month[1]:02d}",
)
fig.tight_layout()
out = os.path.join(OUT_DIR, "07_spatial_sanity.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 6) Temporal sanity

`swapstress.figures.pixel_series.site_series` samples the Level 1 rasters
at the three descriptor sites — a humid, an arid, and a transitional pixel —
opening one raster at a time rather than stacking the record. It reruns stage
06's own `interpolate_pixel` on those three series, so the Level 2 line it
returns is the gap fill, computed for three pixels instead of six million. It
also returns an `observed` mask and a `clamped` mask marking the ends of the
record, where a fill would be extrapolation.

Expect a saw-tooth: sharp wetting after rain, slow logarithmic drydown. A flat
line would mean the model is reading the static covariates and ignoring theta.

In [ ]:
from swapstress.figures.pixel_series import site_series
from swapstress.units import log10_suction_cm_to_mpa

span, series = site_series(LEVEL1_DIR, prefix=PREFIX)

fig, axes = plt.subplots(
    len(series), 1, figsize=(14, 3.2 * len(series)), sharex=True, dpi=120
)
for ax, s in zip(np.atleast_1d(axes), series):
    ax.plot(span, s["filled"], color="tomato", lw=0.8, label="Level 2 (gap-filled)")
    ax.plot(
        span,
        np.where(s["observed"], s["raw"], np.nan),
        ".",
        ms=2.5,
        color="steelblue",
        label="Level 1 (retrieved)",
    )
    if s["clamped"].any():
        ax.fill_between(
            span,
            *ax.get_ylim(),
            where=s["clamped"],
            color="0.85",
            zorder=0,
            label="outside the pixel's record",
        )
    ax.set_ylabel("log10 suction (cm)", fontsize=8)
    ax.set_title(
        f"{s['region']} — {s['place']} ({s['lat']:+.1f}, {s['lon']:+.1f})", fontsize=10
    )
    ax.legend(fontsize=8, loc="upper right")

np.atleast_1d(axes)[-1].set_xlabel("Date")
fig.tight_layout()
out = os.path.join(OUT_DIR, "07_temporal_sanity.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

# The same series in matric potential, which is what the descriptor reports.
sample = series[0]["filled"]
print("\nlog10 suction (cm) -> matric potential (MPa):")
for v in [1.0, 2.0, 3.0, 4.18]:
    print(f"  {v:>5.2f}  ->  {log10_suction_cm_to_mpa(v):+.4f} MPa")
print(
    f"site 0 median: {np.nanmedian(sample):.2f} log10 cm = "
    f"{log10_suction_cm_to_mpa(np.nanmedian(sample)):+.4f} MPa"
)

## 7) Stage 07 — the released bands

Stage 07 turns a day of model output into the released band stack:

```bash
uv run swapstress-package --source-dir <level2 dir> --output-dir <release dir> \
    --level 2 --level1-dir <level1 dir> --container netcdf --drop-linear-suction
```

`--level 2` adds the `gapfill_flag` band, derived by comparing Level 2 against
Level 1, which is why `--level1-dir` is required there. `--container netcdf`
writes time-stacked CF files (the archive of record); `geotiff` writes one file
per day.

`build_bands` is the pure computation behind it — no I/O, no container
assumptions. The three suction representations are exact transforms of one
another; `--drop-linear-suction` drops the one that compresses worst.

In [ ]:
from swapstress.inference.product import build_bands, derive_gapfill_flag

if show_day in level2:
    day_arr, extent = read_day(level2[show_day])
    # Level 2 packaging: the flag is recovered by comparing the two levels,
    # because the stage records whether a whole day was filled, not which pixels.
    flag = derive_gapfill_flag(np.nan_to_num(day_arr, nan=-9999.0), level1[show_day])
else:
    day_arr, extent = read_day(level1[show_day])
    flag = None  # Level 1: nothing was filled, so there is no flag band.

bands = build_bands(day_arr, gapfill_flag=flag, include_linear_suction=True)

print(f"{len(bands.specs)} bands for {show_day}:")
for spec in bands.specs:
    arr = bands.band(spec.name)
    valid = arr[bands.valid]
    print(
        f"  {spec.name:<22s} {spec.units:<12s} [{valid.min():.4g}, {valid.max():.4g}]"
    )

print(f"\nvalid pixels: {int(bands.valid.sum()):,}")
if flag is not None:
    filled = int((bands.band("gapfill_flag")[bands.valid] == 1).sum())
    print(f"gap-filled:   {filled:,}")

In [ ]:
ncols = 2 if flag is not None else 1
fig, axes = plt.subplots(1, ncols, figsize=(8 * ncols, 5), dpi=120, squeeze=False)

mpa = bands.band("matric_potential_MPa").copy()
mpa[~bands.valid] = np.nan
draw_map(
    axes[0][0],
    mpa,
    extent,
    cmap="RdBu",
    label="matric potential (MPa)",
    title=f"matric_potential_MPa — {show_day}",
)

if flag is not None:
    flag_arr = bands.band("gapfill_flag").copy()
    flag_arr[~bands.valid] = np.nan
    draw_map(
        axes[0][1],
        flag_arr,
        extent,
        cmap="coolwarm",
        vmin=0,
        vmax=1,
        label="0 = retrieved, 1 = gap-filled",
        title=f"gapfill_flag — {show_day}",
    )

fig.tight_layout()
out = os.path.join(OUT_DIR, "07_released_bands.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## Next

`08_validation.ipynb` is stage 04: how well the model that produced these
rasters does against held-out observations.